# Phase 6 DAEAC + ADDA Kaggle Driver

Thin driver notebook. Training logic lives in repository scripts.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIG = 'configs/phase6_daeac_adda.yaml'
METHOD = 'adda'
PAIRS = ['ds1_ds2', 'ds1_incart', 'ds1_svdb', 'mitbih_incart', 'mitbih_svdb']
EVAL_DATASET = 'target'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Repo path:', ECG)

## 1. Clone Or Locate Repo

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
os.chdir(ECG)
print(Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy Phase 6 Data And Base Checkpoint

In [ ]:
data_dirs = [p for p in Path('/kaggle/input').glob('**/daeac') if p.is_dir()]
DATA_DIR = data_dirs[0] if data_dirs else Path('/kaggle/input/YOUR_DATASET/daeac')
DEST = Path('data/processed/phase6_daeac_paper')
DEST.mkdir(parents=True, exist_ok=True)
if not DATA_DIR.exists():
    raise FileNotFoundError('Could not find a daeac data folder under /kaggle/input. Edit DATA_DIR manually.')
for p in DATA_DIR.glob('*.npz'):
    shutil.copy2(p, DEST / p.name)
    print('copied data', p.name)

ckpt_candidates = list(Path('/kaggle/input').glob('**/daeac_base_best.pt'))
if not ckpt_candidates:
    raise FileNotFoundError('Attach daeac_base_best.pt as a Kaggle input.')
BASE_CKPT = Path('outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt')
BASE_CKPT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(ckpt_candidates[0], BASE_CKPT)
print('base checkpoint:', BASE_CKPT)
!ls -lh data/processed/phase6_daeac_paper outputs/phase6_daeac_paper/checkpoints

## 4. Validate

In [ ]:
!python scripts/check_repo.py
base_jobs = [('outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt', 'ds1'), ('outputs/phase6_daeac_mitbih_base/checkpoints/daeac_base_mitbih_best.pt', 'mitbih')]
for checkpoint, corpus in base_jobs:
    if not Path(checkpoint).exists():
        subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/01_train_base.py', '--config', 'configs/phase6_daeac_paper.yaml', '--source-corpus', corpus], check=True)
for pair in PAIRS:
    subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/00_validate_data.py', '--config', CONFIG, '--domain-pair', pair], check=True)

## 5. Smoke Run

In [ ]:
subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/01_train_adda.py', '--config', CONFIG, '--domain-pair', 'ds1_ds2', '--epochs', '1', '--max-source-samples', '512', '--max-target-samples', '512', '--max-val-samples', '512', '--checkpoint-prefix', 'daeac_adda_ds1_ds2_smoke'], check=True)

## 6. Full Adaptation Run

In [ ]:
for pair in PAIRS:
    subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/01_train_adda.py', '--config', CONFIG, '--domain-pair', pair], check=True)

## 7. Evaluate And Report

In [ ]:
for pair in PAIRS:
    prefix = f'daeac_adda_{pair}'
    checkpoint = f'outputs/phase6_daeac_adda_{pair}/checkpoints/{prefix}_latest.pt'
    subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/02_eval.py', '--config', CONFIG, '--domain-pair', pair, '--checkpoint', checkpoint, '--method-name', f'{prefix}_latest', '--dataset', EVAL_DATASET], check=True)
    subprocess.run([sys.executable, 'scripts/phase6_daeac_adversarial/03_make_report.py', '--config', CONFIG, '--domain-pair', pair, '--method-name', prefix], check=True)

## 8. Save Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_adda_outputs.zip outputs/phase6_daeac_adda_* outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt outputs/phase6_daeac_mitbih_base/checkpoints/daeac_base_mitbih_best.pt